# Step 3 Runner v3 — static file check + optional diagnostic run

This notebook does **not** edit source code. It only pulls your public GitHub repo, checks required Python/shell files, optionally prepares RegDB, and optionally runs relation diagnostics.

It intentionally does **not** check Markdown/docs files.


In [8]:
CFG = {
    "GITHUB_USERNAME": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",
    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "REGDB_SOURCE": "/kaggle/input/datasets/xqq027/reg-db/RegDB",  # empty = auto-detect by prepare_regdb_kaggle.py
    "USE_GITHUB_TOKEN": False,  # repo is public now; set True only if you make it private again
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",
    "RUN_INSTALL": True,
    "RUN_DATASET_CHECK": True,
    "RUN_DIAGNOSTIC": True,
    "RUN_NAME": "relation_diag_regdb_smoke",
    "DEVICE": "0",
    "SEED": "1",
}
CFG


{'GITHUB_USERNAME': 'TranTruongMMCII',
 'REPO_NAME': 'UIT.CS2309',
 'BRANCH': 'feature/upr-cre',
 'WORK_DIR': '/kaggle/working',
 'DATA_ROOT': '/kaggle/working/VIREID_Dataset',
 'REGDB_SOURCE': '/kaggle/input/datasets/xqq027/reg-db/RegDB',
 'USE_GITHUB_TOKEN': False,
 'GITHUB_TOKEN_SECRET': 'GITHUB_TOKEN',
 'RUN_INSTALL': True,
 'RUN_DATASET_CHECK': True,
 'RUN_DIAGNOSTIC': True,
 'RUN_NAME': 'relation_diag_regdb_smoke',
 'DEVICE': '0',
 'SEED': '1'}

In [9]:
from pathlib import Path
import os
import subprocess
import textwrap

work_dir = Path(CFG["WORK_DIR"])
repo_dir = work_dir / CFG["REPO_NAME"]
repo_url = f"https://github.com/{CFG['GITHUB_USERNAME']}/{CFG['REPO_NAME']}.git"

env = os.environ.copy()

if CFG.get("USE_GITHUB_TOKEN", False):
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(CFG["GITHUB_TOKEN_SECRET"])
    askpass = work_dir / "git_askpass.sh"
    askpass.write_text(textwrap.dedent("""\
    #!/bin/sh
    case "$1" in
      *Username*) echo "{username}" ;;
      *Password*) echo "{token}" ;;
      *) echo "" ;;
    esac
    """.format(username=CFG["GITHUB_USERNAME"], token=token)))
    askpass.chmod(0o700)
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", CFG["BRANCH"], repo_url, str(repo_dir)], check=True, env=env)
else:
    subprocess.run(["git", "fetch", "origin", CFG["BRANCH"]], cwd=repo_dir, check=True, env=env)
    subprocess.run(["git", "checkout", CFG["BRANCH"]], cwd=repo_dir, check=True, env=env)
    subprocess.run(["git", "reset", "--hard", f"origin/{CFG['BRANCH']}"], cwd=repo_dir, check=True, env=env)

print("Repo:", repo_dir)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir, check=True)
subprocess.run(["git", "status", "--short"], cwd=repo_dir, check=True)


Your branch is up to date with 'origin/feature/upr-cre'.
HEAD is now at 0caf458 Add seed parameter and refactor script arguments
Repo: /kaggle/working/UIT.CS2309
0caf458


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD
Already on 'feature/upr-cre'


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [10]:
from pathlib import Path

required_files = [
    "WSL_ReID/main.py",
    "WSL_ReID/wsl.py",
    "WSL_ReID/task/train.py",
    "WSL_ReID/relation_metrics.py",
    "WSL_ReID/requirements-kaggle.txt",
    "WSL_ReID/scripts/apply_kaggle_compat_patches.py",
    "WSL_ReID/scripts/prepare_regdb_kaggle.py",
    "WSL_ReID/scripts/check_kaggle_env.py",
    "WSL_ReID/scripts/run_regdb_relation_diagnostics.sh",
    "WSL_ReID/scripts/collect_relation_stats.py",
]

missing = [p for p in required_files if not (repo_dir / p).exists()]
if missing:
    raise FileNotFoundError("Missing required Step 3 files:\n" + "\n".join(missing))

print("Required source/script files exist. Markdown/docs are not checked.")
for p in required_files:
    print("OK", p)


Required source/script files exist. Markdown/docs are not checked.
OK WSL_ReID/main.py
OK WSL_ReID/wsl.py
OK WSL_ReID/task/train.py
OK WSL_ReID/relation_metrics.py
OK WSL_ReID/requirements-kaggle.txt
OK WSL_ReID/scripts/apply_kaggle_compat_patches.py
OK WSL_ReID/scripts/prepare_regdb_kaggle.py
OK WSL_ReID/scripts/check_kaggle_env.py
OK WSL_ReID/scripts/run_regdb_relation_diagnostics.sh
OK WSL_ReID/scripts/collect_relation_stats.py


In [11]:
# Static consistency checks only for executable/source files.
# These checks prevent known Step 2/Step 3 argument mismatches before running a long job.

checks = []

def contains(path, marker):
    text = (repo_dir / path).read_text(encoding="utf-8")
    return marker in text

# main.py CLI flags
checks.append(("main.py has --save-relation-stats", contains("WSL_ReID/main.py", "--save-relation-stats")))
checks.append(("main.py has --relation-stats-dir", contains("WSL_ReID/main.py", "--relation-stats-dir")))
checks.append(("main.py has --relation-stats-every", contains("WSL_ReID/main.py", "--relation-stats-every")))

# train.py diagnostic call
checks.append(("train.py imports save_relation_diagnostics", contains("WSL_ReID/task/train.py", "save_relation_diagnostics")))
checks.append(("train.py reads args.save_relation_stats", contains("WSL_ReID/task/train.py", "save_relation_stats")))

# wsl.py diagnostic state
checks.append(("wsl.py stores last_rgb_score", contains("WSL_ReID/wsl.py", "last_rgb_score")))
checks.append(("wsl.py stores last_ir_score", contains("WSL_ReID/wsl.py", "last_ir_score")))
checks.append(("wsl.py stores rgb_gt", contains("WSL_ReID/wsl.py", "rgb_gt")))
checks.append(("wsl.py stores ir_gt", contains("WSL_ReID/wsl.py", "ir_gt")))

# Step 2a / Step 3 script argument consistency
prepare = (repo_dir / "WSL_ReID/scripts/prepare_regdb_kaggle.py").read_text(encoding="utf-8")
run_script = (repo_dir / "WSL_ReID/scripts/run_regdb_relation_diagnostics.sh").read_text(encoding="utf-8")
collect = (repo_dir / "WSL_ReID/scripts/collect_relation_stats.py").read_text(encoding="utf-8")

checks.append(("prepare_regdb_kaggle.py supports --data-root", "--data-root" in prepare))
checks.append(("prepare_regdb_kaggle.py supports --regdb-source", "--regdb-source" in prepare))
checks.append(("run_regdb_relation_diagnostics.sh uses --data-root", "--data-root" in run_script))
checks.append(("run_regdb_relation_diagnostics.sh uses --regdb-source or PREPARE_ARGS", ("--regdb-source" in run_script or "PREPARE_ARGS" in run_script)))
checks.append(("run_regdb_relation_diagnostics.sh does not use old --source", "--source" not in run_script))
checks.append(("run_regdb_relation_diagnostics.sh does not use old --output", "--output" not in run_script))
checks.append(("run_regdb_relation_diagnostics.sh does not use old --input-root", "--input-root" not in run_script))
checks.append(("collect_relation_stats.py supports --stats-dir", "--stats-dir" in collect))
checks.append(("collect_relation_stats.py has a CSV output option", ("--output" in collect or "--csv-output" in collect)))

failed = [name for name, ok in checks if not ok]
for name, ok in checks:
    print(("OK   " if ok else "FAIL ") + name)

if failed:
    raise RuntimeError("Static file consistency failed:\n" + "\n".join(failed))

print("Static file consistency check passed.")


OK   main.py has --save-relation-stats
OK   main.py has --relation-stats-dir
OK   main.py has --relation-stats-every
OK   train.py imports save_relation_diagnostics
OK   train.py reads args.save_relation_stats
OK   wsl.py stores last_rgb_score
OK   wsl.py stores last_ir_score
OK   wsl.py stores rgb_gt
OK   wsl.py stores ir_gt
OK   prepare_regdb_kaggle.py supports --data-root
OK   prepare_regdb_kaggle.py supports --regdb-source
OK   run_regdb_relation_diagnostics.sh uses --data-root
OK   run_regdb_relation_diagnostics.sh uses --regdb-source or PREPARE_ARGS
OK   run_regdb_relation_diagnostics.sh does not use old --source
OK   run_regdb_relation_diagnostics.sh does not use old --output
OK   run_regdb_relation_diagnostics.sh does not use old --input-root
OK   collect_relation_stats.py supports --stats-dir
OK   collect_relation_stats.py has a CSV output option
Static file consistency check passed.


In [12]:
# Syntax check only. This still does not run training.
cmd = [
    "python", "-m", "py_compile",
    "WSL_ReID/main.py",
    "WSL_ReID/wsl.py",
    "WSL_ReID/task/train.py",
    "WSL_ReID/relation_metrics.py",
    "WSL_ReID/scripts/collect_relation_stats.py",
    "WSL_ReID/scripts/prepare_regdb_kaggle.py",
    "WSL_ReID/scripts/check_kaggle_env.py",
    "WSL_ReID/scripts/apply_kaggle_compat_patches.py",
]
subprocess.run(cmd, cwd=repo_dir, check=True)
print("py_compile passed.")


py_compile passed.


In [13]:
# Optional: install minimal packages and check RegDB layout.
if CFG.get("RUN_INSTALL", True):
    subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=repo_dir / "WSL_ReID", check=True)

if CFG.get("RUN_DATASET_CHECK", False):
    wsl_dir = repo_dir / "WSL_ReID"
    subprocess.run(["python", "scripts/apply_kaggle_compat_patches.py"], cwd=wsl_dir, check=True)
    prepare_cmd = ["python", "scripts/prepare_regdb_kaggle.py", "--data-root", CFG["DATA_ROOT"]]
    if CFG.get("REGDB_SOURCE"):
        prepare_cmd += ["--regdb-source", CFG["REGDB_SOURCE"]]
    subprocess.run(prepare_cmd, cwd=wsl_dir, check=True)
    subprocess.run(["python", "scripts/check_kaggle_env.py", "--data-root", CFG["DATA_ROOT"]], cwd=wsl_dir, check=True)
else:
    print("RUN_DATASET_CHECK=False, skipping dataset/GPU check.")


No compatibility patches were needed.
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male_back_v_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 train_thermal_1.txt
first line: Thermal/285/male_back_t_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Thermal/285/male_back_t_05528_285.bmp
exists: True
image size: (64, 128

In [14]:
# Optional: run the diagnostic smoke test.
if CFG.get("RUN_DIAGNOSTIC", False):
    env_run = os.environ.copy()
    env_run["DATA_ROOT"] = CFG["DATA_ROOT"]
    env_run["REGDB_SOURCE"] = CFG.get("REGDB_SOURCE", "")
    env_run["RUN_NAME"] = CFG["RUN_NAME"]
    env_run["DEVICE"] = CFG["DEVICE"]
    env_run["SEED"] = CFG["SEED"]
    subprocess.run(["bash", "scripts/run_regdb_relation_diagnostics.sh"], cwd=repo_dir / "WSL_ReID", check=True, env=env_run)

    stats_dir = repo_dir / "WSL_ReID" / ".." / "saved_regdb_resnet" / f"{CFG['RUN_NAME']}_1" / "relation_stats"
    stats_dir = stats_dir.resolve()
    print("Stats dir:", stats_dir)
    print("Files:")
    for p in sorted(stats_dir.glob("*")):
        print(" -", p)
else:
    print("RUN_DIAGNOSTIC=False, static checks only.")


RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male_back_v_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 train_thermal_1.txt
first line: Thermal/285/male_back_t_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Thermal/285/male_back_t_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 test_visible_1.

100%|██████████| 97.8M/97.8M [00:00<00:00, 197MB/s] 


valid model file not existed!
from 0 epoch training
Time: 2026-06-12 06:14:57 | start phase1 from epoch 0
extracting rgb features
extracting ir features
get match labels
[relation diagnostics] epoch=0 common=0 specific=45 remain=7 mutual=0.0000 r2i_acc=0.0000 i2r_acc=0.0000 common_acc=0.0000
Time: 2026-06-12 06:24:24 | phase 1 epoch 1; Setting: ../saved_regdb_resnet/relation_diag_regdb_smoke_1
e_lr: 4.5e-06
r2r_id_loss: 4.8945084;  i2i_id_loss: 4.866123;  
r2r_tri_loss: 0.722459;  i2i_tri_loss: 0.6957898;  

R1:0.0597;   R10:0.1549;  R20:0.2233;  mAP:0.0767;  mINP:0.0505
                       Best_R1: 0.0597;    Best mAP: 0.0767
make dir ../saved_pretrain_regdb_resnet_1_../saved_regdb_resnet/relation_diag_regdb_smoke_1/ successful!
Time: 2026-06-12 06:24:24 | start phase2 from epoch 0
extracting rgb features
extracting ir features
get match labels
[relation diagnostics] epoch=0 common=5 specific=43 remain=36 mutual=0.1087 r2i_acc=0.1522 i2r_acc=0.0667 common_acc=0.2000
Epoch: 0;Time: 